In [1]:
import numpy as np
import pandas as pd

In [2]:
# パリソンデータから外径/板厚情報を抽出
def extract_input_data(lines: list):
    data = []

    for line in lines:
        values = line.split()
        data.append([float(value) for value in values])

    return data

In [3]:
# Ls-dyna出力データから*ELEMENT_SHELL_THICKNESSの情報を抽出
def extract_result_data(lines: list):
    data = []
    extract = False

    for line in lines:
        if line.startswith("$"):
            continue
        elif line.startswith("*ELEMENT_SHELL_THICKNESS"):
            extract = True
            continue
        elif line.startswith("*"):
            extract = False

        if extract:
            values = line.split()
            data.append([float(value) for value in values])

    id_list_list = [sublist for index, sublist in enumerate(data) if index % 2 == 0]
    id_list = [sublist[0] for sublist in id_list_list]

    result_list_list = [sublist for index, sublist in enumerate(data) if index % 2 != 0]
    result_list = [sum(sublist) / len(sublist) for sublist in result_list_list]

    result_data_list = [[x, y] for x, y in zip(id_list, result_list)]

    return result_data_list

In [4]:
# パリソンデータのみをフィルタリング
def filter_data(data: list):
    filtered_data = []

    for entry in data:
        if 500000 <= entry[0] <= 699999:
            filtered_data.append(entry)

    return filtered_data

In [5]:
ntr = 10  # 教師データ数
epoch = 100  # エポック数
step = ntr * epoch  # ステップ数
meshx = 100  # x方向メッシュ数
meshy = 100  # y方向メッシュ数
meshcut = meshx // 4  # メッシュ展開位置
meshhalf = meshx // 2  # 2分割数
setdata = np.empty([0, meshy + 6, meshx * 2 + 12, 1])

# パリソンデータの読み込み
# filepr = "/tmp/traindata/parison" + str(k + 1) + ".dat"
filepr = "../data/parison1.dat"
with open(filepr, "r") as f:
    lines = f.readlines()

input_data = []
for line in lines:
    values = line.split()
    input_data.append([float(value) for value in values])

print(len(input_data))

del input_data[0:1]
input_data = np.repeat(input_data, meshhalf).reshape(meshy, meshx)
# display(pd.DataFrame(input_data).head())
input_data[:, :meshhalf] = (input_data[:, :meshhalf] - 330) / 18
input_data[:, meshhalf:] = input_data[:, meshhalf:] - 11
input_data = np.pad(input_data, [(3, 3), (3, 3)], "constant")
df_input_data = (pd.DataFrame(input_data))
print(df_input_data.shape)
df_input_data.head()

101
(106, 106)


,0,1,2,3,4,5,6,7,8,9,...,96,97,98,99,100,101,102,103,104,105
0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.118565,0.118565,0.118565,0.118565,0.118565,0.118565,0.118565,...,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.0,0.0,0.0
4,0.0,0.0,0.0,0.142782,0.142782,0.142782,0.142782,0.142782,0.142782,0.142782,...,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.0,0.0,0.0


In [6]:
# LS-dyna出力ファイルの読み込み
# filedn = "/tmp/traindata/lsdyna" + str(k + 1) + ".k"
filedn = "../data/lsdyna1.k"
with open(filedn, "r") as f:
    lines = f.readlines()

result_data_list = []
extract = False

for line in lines:
    if line.startswith("$"):
        continue
    elif line.startswith("*ELEMENT_SHELL_THICKNESS"):
        extract = True
        continue
    elif line.startswith("*"):
        extract = False

    if extract:
        values = line.split()
        result_data_list.append([float(value) for value in values])

id_list_list = [sublist for index, sublist in enumerate(result_data_list) if index % 2 == 0]
# display(pd.DataFrame(id_list_list).head(3))
id_list = [sublist[0] for sublist in id_list_list]
# display(pd.DataFrame(id_list, columns=['ele_id']).head(3))

result_list_list = [sublist for index, sublist in enumerate(result_data_list) if index % 2 != 0]
# display(pd.DataFrame(result_list_list).head(3))
result_list = [sum(sublist) / len(sublist) for sublist in result_list_list]
# display(pd.DataFrame(result_list, columns=['thickness']).head(3))

result_data_list = [[x, y] for x, y in zip(id_list, result_list)]

len(result_data_list)
# pd.DataFrame(result_data_list, columns=['ele_id', 'thickness']).head(3)

10000

In [7]:
# 500000 - 699999 にフィルター
filtered_result_list = filter_data(result_data_list)

result_data = [sublist[1] for sublist in filtered_result_list]
result_data = np.reshape(result_data, (meshy, meshx))
df_result_data = pd.DataFrame(result_data)
display(df_result_data.head())
result_data = np.block(
    [result_data[:, meshx - meshcut :], result_data[:, : meshx - meshcut]]
)
df_result_data = pd.DataFrame(result_data)
display(df_result_data.head())
result_data = (result_data - 10) / 10
result_data = np.pad(result_data, [(3, 3), (3, 3)], "constant")
df_result_data = pd.DataFrame(result_data)
df_result_data.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,11.80646,11.80085,11.79130,11.78032,11.77070,11.76720,11.76967,11.77759,11.78868,11.80059,...,11.80052,11.78864,11.77741,11.76950,11.76694,11.77059,11.78022,11.79144,11.80091,11.80641
1,11.75431,11.76092,11.76943,11.77627,11.77933,11.77970,11.78000,11.77703,11.77147,11.76363,...,11.76353,11.77134,11.77690,11.77958,11.77943,11.77924,11.77622,11.76954,11.76105,11.75433
2,11.73452,11.74269,11.75398,11.76469,11.77168,11.77431,11.77434,11.76737,11.75576,11.74266,...,11.74242,11.75578,11.76711,11.77401,11.77415,11.77146,11.76461,11.75409,11.74306,11.73459
3,11.72343,11.73047,11.74077,11.75105,11.75799,11.76204,11.76119,11.75424,11.74202,11.72837,...,11.72832,11.74192,11.75408,11.76092,11.76179,11.75793,11.75098,11.74092,11.73066,11.72353
4,11.71224,11.71852,11.72751,11.73637,11.74315,11.74695,11.74608,11.73967,11.72845,11.71652,...,11.71637,11.72832,11.73962,11.74611,11.74694,11.74304,11.73638,11.72762,11.71862,11.71223


,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,11.75020,11.75227,11.75670,11.76562,11.77700,11.78720,11.79532,11.80316,11.80666,11.80804,...,11.80631,11.80510,11.80159,11.79401,11.78598,11.77625,11.76536,11.75635,11.75263,11.75048
1,11.81305,11.80994,11.80473,11.79540,11.78261,11.76975,11.76432,11.76909,11.77889,11.78373,...,11.78308,11.77840,11.76888,11.76406,11.76950,11.78230,11.79521,11.80466,11.80991,11.81300
2,11.79632,11.79638,11.79422,11.78603,11.77173,11.75685,11.74966,11.75210,11.75936,11.76315,...,11.76350,11.75988,11.75275,11.75035,11.75733,11.77175,11.78575,11.79392,11.79571,11.79610
3,11.76925,11.76920,11.76830,11.76252,11.75307,11.74337,11.73841,11.74077,11.74509,11.74724,...,11.74823,11.74660,11.74247,11.73998,11.74452,11.75334,11.76214,11.76736,11.76876,11.76908
4,11.74870,11.74648,11.74268,11.73814,11.73205,11.72778,11.72660,11.72906,11.73298,11.73416,...,11.73578,11.73536,11.73178,11.72899,11.72944,11.73267,11.73778,11.74215,11.74633,11.74875


,0,1,2,3,4,5,6,7,8,9,...,96,97,98,99,100,101,102,103,104,105
0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
1,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
2,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
3,0.0,0.0,0.0,0.175020,0.175227,0.175670,0.176562,0.177700,0.178720,0.179532,...,0.179401,0.178598,0.177625,0.176536,0.175635,0.175263,0.175048,0.0,0.0,0.0
4,0.0,0.0,0.0,0.181305,0.180994,0.180473,0.179540,0.178261,0.176975,0.176432,...,0.176406,0.176950,0.178230,0.179521,0.180466,0.180991,0.181300,0.0,0.0,0.0


In [8]:
df_input_data.head()

,0,1,2,3,4,5,6,7,8,9,...,96,97,98,99,100,101,102,103,104,105
0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.118565,0.118565,0.118565,0.118565,0.118565,0.118565,0.118565,...,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.0,0.0,0.0
4,0.0,0.0,0.0,0.142782,0.142782,0.142782,0.142782,0.142782,0.142782,0.142782,...,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.0,0.0,0.0


In [9]:
data = np.block([input_data, result_data])
data.shape

(106, 212)

In [10]:
data = np.block([input_data, result_data]).reshape(1, meshy + 6, meshx * 2 + 12, 1)
setdata = np.concatenate([setdata, data], 0).astype("float32")

print(data.shape)
setdata.shape

(1, 106, 212, 1)


(1, 106, 212, 1)